In [5]:
"""
Entity TYPE resolution, STEP 2: label the (reviewed/edited) clusters.

Not to be confused with step2_validate_entity_groups.py (which resolves
entity NAMES). This resolves entity TYPES for the 'other'-typed
entities from step1_generate_type_clusters.py.

Two changes from the earlier version, both from real review feedback:

  1. CLOSED taxonomy. The earlier version let the LLM invent a new
     category whenever none of the existing ones felt like a perfect
     fit, which produced dozens of one-off near-duplicate categories
     (protein_band, protein_band_intensity, molecular_interaction,
     yield_metric, yield_property, extraction_yield, extraction_metric,
     ...), most of them sub-categories of a small number of real types
     rather than genuinely new types. TYPE_CATEGORIES below is now
     fixed, 8 categories, no new ones allowed, and deliberately no
     'other' escape valve either (an earlier "other" fallback absorbed
     more than half the corpus, defeating the point of a taxonomy).
     The model must pick its single best fit even when imperfect.

  2. Singletons now get classified too. The earlier version skipped
     any entity with no cluster partner, leaving it permanently
     untyped, that's not workable for a final KG schema, every entity
     needs a type whether or not it has a synonym-cluster partner.
     Singletons are classified in batches, one type per entity, no
     coherence question involved since there's nothing to agree or
     disagree about for a single entity.

Requires OPENAI_API_KEY as an environment variable.
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
REVIEWED_CLUSTERS_PATH = "entity_type_clusters_for_review.xlsx"  # edit this file by hand first if needed
REVIEWED_CLUSTERS_SHEET = "clusters_for_review"  # step1 now writes READ_ME_FIRST as the first sheet
LLM_MODEL = "gpt-5.6-sol"
SINGLETON_BATCH_SIZE = 25  # entities classified per LLM call for singletons

OUTPUT_REVIEW_XLSX = "entity_type_resolution_review.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Closed taxonomy, no new categories, no 'other'. The model must pick
# its single best fit from this fixed list every time.
TYPE_CATEGORIES = [
    "functional_property", "physicochemical_property", "structural_property",
    "modification_method", "extraction_method", "material_used_directly",
    "rheological_property", "sensory_property",
]

# Domain-specific mapping guidance, confirmed from real review of this
# corpus: spectroscopic/analytical characterization (FTIR, amide bands,
# NMR relaxation, XRD, secondary structure spectral features) is used
# here for STRUCTURAL analysis, so it belongs under structural_property,
# not physicochemical_property. Gelling, viscosity, texture, and
# mechanical behavior (modulus, viscoelasticity, yield point, foam
# firmness) all belong under rheological_property, a dedicated
# category, not physicochemical_property either.
DOMAIN_GUIDANCE = """Domain-specific mapping rules for this corpus, apply these consistently:
- Spectroscopic, molecular or analytical structural characterization (FTIR peaks, amide band
  intensities, NMR relaxation times/peaks, XRD, secondary structure proportions like
  alpha-helix/beta-sheet/beta-turn content) -> structural_property, NOT physicochemical_property.
- Gelling, viscosity, texture, and mechanical behavior (storage/loss modulus, yield point,
  viscoelasticity, foam firmness/density/compactness, mechanical strength) -> rheological_property,
  NOT physicochemical_property.
- Flavor, aroma, taste, and color tied to sensory perception -> sensory_property.
"""

# ---------------------------------------------------------------
# LOAD REVIEWED CLUSTERS
# ---------------------------------------------------------------
cluster_df = pd.read_excel(REVIEWED_CLUSTERS_PATH, sheet_name=REVIEWED_CLUSTERS_SHEET)
print(f"Loaded {len(cluster_df)} entities across {cluster_df['cluster_id'].nunique()} clusters")

cluster_sizes = cluster_df.groupby("cluster_id").size()
multi_member_ids = cluster_sizes[cluster_sizes >= 2].index
singleton_ids = cluster_sizes[cluster_sizes < 2].index
print(f"Multi-member clusters: {len(multi_member_ids)}, singletons: {len(singleton_ids)}")

# ---------------------------------------------------------------
# LLM: PROPOSE A CATEGORY LABEL PER MULTI-MEMBER CLUSTER, CLOSED TAXONOMY
# ---------------------------------------------------------------
LABEL_PROMPT = """You are assigning entity types for a plant protein functional properties
knowledge graph, using a FIXED, CLOSED taxonomy. You must choose exactly one type from
this list, no exceptions, no new categories, no "other":
{categories}

{domain_guidance}
Below is a cluster of entity names that were grouped together (embedding similarity,
then manually reviewed/adjusted by the researcher). Evaluate this cluster EXACTLY as
given, do not split or regroup it.

Entities in this cluster:
{entities}

Decide:
1. Do ALL of these entities genuinely share one type from the list above? Set coherent
   to true/false.
2. Regardless of your answer to (1), you MUST choose exactly one type from the fixed
   list, either for the whole cluster (if coherent) or your best-guess single best fit
   for what the majority/core entities share (if not coherent). Never leave this blank,
   never invent a category not in the list.
3. One-sentence justification. If coherent is false, name SPECIFICALLY which entity or
   entities you believe do not belong and why.

Respond ONLY with JSON, no markdown fences:
{{"coherent": true/false, "proposed_type": "...", "justification": "..."}}
"""


def label_cluster(entities):
    prompt = LABEL_PROMPT.format(
        categories=", ".join(TYPE_CATEGORIES),
        domain_guidance=DOMAIN_GUIDANCE,
        entities="\n".join(f"- {e}" for e in entities),
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,
    )
    raw = resp.choices[0].message.content.strip()
    time.sleep(0.2)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        return {"coherent": False, "proposed_type": None, "justification": f"PARSE_FAILED: {raw[:200]}"}

    # proposed_type is mandatory per the contract, even on disagreement,
    # a missing one on "coherent" is treated as a parse problem, not applied
    if result.get("coherent") and not result.get("proposed_type"):
        result["justification"] = ("MISSING_TYPE despite coherent=true: "
                                     + str(result.get("justification", "")))
        result["coherent"] = False
    # enforce the closed taxonomy even if the model drifts from it
    if result.get("proposed_type") not in TYPE_CATEGORIES:
        result["justification"] = (f"INVALID_TYPE '{result.get('proposed_type')}' not in closed "
                                     f"taxonomy: {result.get('justification', '')}")
        result["proposed_type"] = None
        result["coherent"] = False
    return result


multi_results = []
for cid in multi_member_ids:
    entities_in_cluster = cluster_df.loc[cluster_df["cluster_id"] == cid, "entity"].tolist()
    label_result = label_cluster(entities_in_cluster)
    coherent = bool(label_result.get("coherent"))
    multi_results.append({
        "cluster_id": cid,
        "cluster_size": len(entities_in_cluster),
        "entities": "; ".join(entities_in_cluster),
        "coherent": coherent,
        "proposed_type": label_result.get("proposed_type"),
        "justification": label_result.get("justification"),
        "needs_manual_review": not coherent,
    })

review_df = pd.DataFrame(multi_results).sort_values("cluster_size", ascending=False) if multi_results else pd.DataFrame(
    columns=["cluster_id", "cluster_size", "entities", "coherent", "proposed_type", "justification", "needs_manual_review"]
)
n_coherent = review_df["coherent"].fillna(False).sum() if len(review_df) else 0
print(f"\n{n_coherent} multi-member clusters judged coherent, "
      f"{(len(review_df) - n_coherent) if len(review_df) else 0} disagreed")

# ---------------------------------------------------------------
# LLM: CLASSIFY SINGLETONS IN BATCHES, CLOSED TAXONOMY
# no coherence question, one type per entity, independently
# ---------------------------------------------------------------
SINGLETON_PROMPT = """You are assigning entity types for a plant protein functional properties
knowledge graph, using a FIXED, CLOSED taxonomy. For EACH entity below, choose exactly one
type from this list, no exceptions, no new categories, no "other":
{categories}

{domain_guidance}
These entities are independent, unrelated to each other, classify each one on its own merits.

Entities:
{entities}

Respond ONLY with JSON, no markdown fences, a list covering every entity IN THE SAME ORDER:
[{{"entity": "...", "proposed_type": "...", "justification": "..."}}]
"""


def classify_singleton_batch(entities):
    prompt = SINGLETON_PROMPT.format(
        categories=", ".join(TYPE_CATEGORIES),
        domain_guidance=DOMAIN_GUIDANCE,
        entities="\n".join(f"- {e}" for e in entities),
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=1,
    )
    raw = resp.choices[0].message.content.strip()
    time.sleep(0.2)
    try:
        results = json.loads(raw)
    except json.JSONDecodeError:
        return [{"entity": e, "proposed_type": None,
                  "justification": f"PARSE_FAILED: {raw[:200]}"} for e in entities]

    # safety check: same shape guarantee as everywhere else in this
    # pipeline, if the batch doesn't come back 1:1 with what went in,
    # don't trust a partial/misaligned mapping
    returned_entities = [r.get("entity") for r in results]
    if sorted(str(x) for x in returned_entities) != sorted(entities):
        return [{"entity": e, "proposed_type": None,
                  "justification": "BATCH_MISMATCH: response didn't map 1:1 to input"} for e in entities]

    for r in results:
        if r.get("proposed_type") not in TYPE_CATEGORIES:
            r["justification"] = f"INVALID_TYPE '{r.get('proposed_type')}': {r.get('justification', '')}"
            r["proposed_type"] = None
    return results


singleton_entities = cluster_df.loc[cluster_df["cluster_id"].isin(singleton_ids), "entity"].tolist()
singleton_results = []
for i in range(0, len(singleton_entities), SINGLETON_BATCH_SIZE):
    batch = singleton_entities[i:i + SINGLETON_BATCH_SIZE]
    batch_results = classify_singleton_batch(batch)
    singleton_results.extend(batch_results)
    print(f"  classified singleton batch {i // SINGLETON_BATCH_SIZE + 1} "
          f"({i + len(batch)}/{len(singleton_entities)})")

singleton_df = pd.DataFrame(singleton_results)
singleton_df["needs_manual_review"] = singleton_df["proposed_type"].isna()
n_classified = singleton_df["proposed_type"].notna().sum()
print(f"\n{n_classified} of {len(singleton_df)} singletons classified successfully")

# ---------------------------------------------------------------
# ENTITY-LEVEL LOOKUP, MULTI-MEMBER + SINGLETON, COMBINED
# ---------------------------------------------------------------
multi_entity_lookup = cluster_df[cluster_df["cluster_id"].isin(multi_member_ids)].merge(
    review_df[["cluster_id", "proposed_type", "coherent", "justification", "needs_manual_review"]],
    on="cluster_id", how="left"
)

singleton_entity_lookup = singleton_df.rename(columns={"entity": "entity"}).copy()
singleton_entity_lookup["cluster_id"] = None
singleton_entity_lookup["coherent"] = None  # not applicable, single entity, nothing to agree/disagree on

entity_lookup = pd.concat([
    multi_entity_lookup[["entity", "cluster_id", "proposed_type", "coherent", "justification", "needs_manual_review"]],
    singleton_entity_lookup[["entity", "cluster_id", "proposed_type", "coherent", "justification", "needs_manual_review"]],
], ignore_index=True)

# ---------------------------------------------------------------
# SAVE FOR FINAL REVIEW
# ---------------------------------------------------------------
readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    f"This file labels clusters from {REVIEWED_CLUSTERS_PATH}, EXACTLY as you arranged them,",
    "using a CLOSED taxonomy (no new categories invented, no 'other' fallback):",
    "  " + ", ".join(TYPE_CATEGORIES),
    "",
    "- cluster_proposals sheet: one row per MULTI-MEMBER cluster_id (2+ entities).",
    "  - coherent: True only if the LLM agreed every entity shares one type.",
    "  - proposed_type: one of the 8 fixed categories, ALWAYS populated, agreement or not.",
    "  - justification: names specifically which entity the LLM objects to, when incoherent.",
    "  - needs_manual_review: True for any cluster judged not coherent.",
    "",
    "- singleton_classifications sheet: entities with no cluster partner, classified",
    "  independently, one type per entity, no coherence question (nothing to compare against).",
    "  needs_manual_review = True only if classification failed (parse error or batch mismatch).",
    "",
    "- entity_level_lookup sheet: multi-member AND singleton results combined, one row per",
    "  entity, ready to merge onto source_type/target_type by entity name once approved.",
]
readme_df = pd.DataFrame({"": readme_rows})

with pd.ExcelWriter(OUTPUT_REVIEW_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    review_df.to_excel(writer, sheet_name="cluster_proposals", index=False)
    singleton_df.to_excel(writer, sheet_name="singleton_classifications", index=False)
    entity_lookup.to_excel(writer, sheet_name="entity_level_lookup", index=False)

print(f"\nSaved labeled review file to {OUTPUT_REVIEW_XLSX}")
print("Review cluster_proposals and singleton_classifications before applying to your main triples file.")
print("Once approved, join entity_level_lookup back onto source/target_type by entity name.")

Loaded 24 entities across 22 clusters
Multi-member clusters: 1, singletons: 21

1 multi-member clusters judged coherent, 0 disagreed
  classified singleton batch 1 (21/21)

21 of 21 singletons classified successfully

Saved labeled review file to entity_type_resolution_review.xlsx
Review cluster_proposals and singleton_classifications before applying to your main triples file.
Once approved, join entity_level_lookup back onto source/target_type by entity name.


C:\Users\olagunju\AppData\Local\Temp\ipykernel_39808\2446519790.py:258: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  entity_lookup = pd.concat([
